## Results Summary and Hyperparameter Tuning

### Results
| Model | Macro F1 | F1 (weighted) | Accuracy | Notes |
|---|---|---|---|---|
| DistilBERT u6 (all 6 layers) | 0.80 | 0.7915 | 0.7913 | **Best overall** |
| DistilBERT u4 (top 4 layers) | 0.79 | 0.7878 | 0.7875 | Close second |
| DistilBERT u2 (top 2 layers) | 0.77 | 0.7653 | 0.7653 | Good tradeoff |
| XGBoost                      | 0.70 | 0.6972 | 0.6976 | Best classical model |
| Random Forest                | 0.68 | 0.6784 | 0.6801 | Decent baseline |
| DistilBERT u0 (frozen)       | 0.59 | 0.5940 | 0.5966 | Too little fine-tuning |
| LSTM                         | 0.19 | 0.2330 | 0.4045 | Collapsed to neutral (class imbalance issue) |

---

### Key Findings

**Best Model: DistilBERT u6**
- Unfreezing all 6 transformer layers gave the best F1 of **0.7915** and accuracy of **0.7913**
- More trainable parameters (64.4% of total) = better feature learning for this dataset
- Positive class had the best per-class F1 (0.83), negative was slightly weaker (0.80)

**Effect of unfreezing layers (DistilBERT)**
- u0 (frozen): 0.5940 — the pretrained features alone are not enough
- u2: 0.7653 — big jump, top 2 layers already capture a lot
- u4: 0.7878 — further improvement, diminishing returns start here
- u6: 0.7915 — marginal gain over u4, but still the best

**Classical models (TF-IDF based)**
- XGBoost outperformed Random Forest (0.697 vs 0.678)
- Both used improved TF-IDF settings: max_features=50000, ngram_range=(1,3), sublinear_tf=True
- The ceiling for TF-IDF based models on this task appears to be around 0.70

**LSTM — collapsed to predicting neutral**
- Suffered from class imbalance: neutral (11117) >> negative (7781) >> positive (8582)
- Model defaulted to always predicting neutral, resulting in near-zero F1 for negative and positive
- Class weights were applied but the Bidirectional fix was not properly applied in this run
- LSTM results excluded from final comparison

---

### Hyperparameter Choices

**TF-IDF (for RF and XGBoost)**
- `max_features=50000` — wider vocabulary captures more tweet-specific terms
- `ngram_range=(1,3)` — captures unigrams, bigrams, trigrams (e.g. "not good at all")
- `sublinear_tf=True` — log-scales term frequency, reduces dominance of common words
- `min_df=2` — ignores terms appearing only once (noise reduction)

**XGBoost**
- `n_estimators=500` — more trees = better ensemble
- `learning_rate=0.1` — standard learning rate, balances speed and accuracy
- `max_depth=6` — controls tree complexity, prevents overfitting

**DistilBERT**
- `lr=2e-5` — standard fine-tuning learning rate for transformer models
- `epochs=3` — transformers converge fast, 3 epochs is typically sufficient
- `batch_size=32` — balances memory usage and gradient stability
- `max_len=64` — tweets are short, 64 tokens is sufficient coverage

In [25]:
!pip install xgboost lightgbm torch
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score, classification_report

In [26]:
df = pd.read_csv('../data/output.csv')
df = df[['cleaned_text', 'sentiment']].dropna()

le = LabelEncoder()
df['label'] = le.fit_transform(df['sentiment'])
print(df['sentiment'].value_counts())  # check class distribution

sentiment
neutral     11117
positive     8582
negative     7781
Name: count, dtype: int64


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'], df['label'],
    test_size=0.2, random_state=42, stratify=df['label']
)

In [28]:
tfidf = TfidfVectorizer(
    max_features=50000,   # was 10000, more features = more info
    ngram_range=(1, 3),   # was (1,2), captures more phrases
    sublinear_tf=True,    # dampens high frequency words
    min_df=2
)
X_train_tf = tfidf.fit_transform(X_train)
X_test_tf  = tfidf.transform(X_test)

In [29]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train_tf, y_train)
rf_preds = rf.predict(X_test_tf)
print(classification_report(y_test, rf_preds, target_names=le.classes_))

              precision    recall  f1-score   support

    negative       0.75      0.52      0.62      1556
     neutral       0.60      0.79      0.68      2223
    positive       0.78      0.68      0.73      1717

    accuracy                           0.68      5496
   macro avg       0.71      0.66      0.68      5496
weighted avg       0.70      0.68      0.68      5496



In [30]:
from xgboost import XGBClassifier
xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=6,
    eval_metric='mlogloss',
    random_state=42
)

xgb.fit(X_train_tf, y_train)
xgb_preds = xgb.predict(X_test_tf)
print(classification_report(y_test, xgb_preds, target_names=le.classes_))

              precision    recall  f1-score   support

    negative       0.76      0.57      0.65      1556
     neutral       0.62      0.78      0.69      2223
    positive       0.80      0.70      0.75      1717

    accuracy                           0.70      5496
   macro avg       0.72      0.69      0.70      5496
weighted avg       0.71      0.70      0.70      5496



In [31]:
results = {
    "Random Forest": {
        "Accuracy": accuracy_score(y_test, rf_preds),
        "F1 (weighted)": f1_score(y_test, rf_preds, average='weighted')
    },
    "XGBoost": {
        "Accuracy": accuracy_score(y_test, xgb_preds),
        "F1 (weighted)": f1_score(y_test, xgb_preds, average='weighted')
    },
}
print(pd.DataFrame(results).T)

               Accuracy  F1 (weighted)
Random Forest  0.680131       0.678353
XGBoost        0.697598       0.697158


In [32]:
!pip install tensorflow

import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [33]:
MAX_WORDS = 20000   # vocabulary size
MAX_LEN = 50        # max tweet length in words

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post', truncating='post')

print("Train shape:", X_train_pad.shape)
print("Test shape: ", X_test_pad.shape)

Train shape: (21984, 50)
Test shape:  (5496, 50)


In [34]:
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.layers import Bidirectional
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

def build_lstm(units_1=128, units_2=64, dropout=0.3, bidirectional=True):
    """Bidirectional LSTM — fixes collapsed-to-neutral issue."""
    layer_1 = LSTM(units_1, return_sequences=True)
    layer_2 = LSTM(units_2)
    if bidirectional:
        layer_1 = Bidirectional(layer_1)
        layer_2 = Bidirectional(layer_2)
    model = Sequential([
        Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),
        layer_1,
        Dropout(dropout),
        layer_2,
        Dropout(dropout),
        Dense(64, activation='relu'),
        Dense(3, activation='softmax')
    ])
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

lstm_model = build_lstm(bidirectional=True)
lstm_model.summary()


Class weights: {0: np.float64(1.1771887550200804), 1: np.float64(0.8239262424106139), 2: np.float64(1.067443554260743)}


/opt/miniconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_17"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_17 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_20                │ ?                      │   0 (unbuilt) │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_34 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_21                │ ?                      │   0 (unbuilt) │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_35 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [35]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = lstm_model.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/10
619/619 ━━━━━━━━━━━━━━━━━━━━ 43s 64ms/step - accuracy: 0.6120 - loss: 0.8108 - val_accuracy: 0.6944 - val_loss: 0.7063
Epoch 2/10
619/619 ━━━━━━━━━━━━━━━━━━━━ 42s 68ms/step - accuracy: 0.7784 - loss: 0.5415 - val_accuracy: 0.7140 - val_loss: 0.6986
Epoch 3/10
619/619 ━━━━━━━━━━━━━━━━━━━━ 44s 72ms/step - accuracy: 0.8537 - loss: 0.3804 - val_accuracy: 0.7003 - val_loss: 0.7968
Epoch 4/10
619/619 ━━━━━━━━━━━━━━━━━━━━ 43s 70ms/step - accuracy: 0.8996 - loss: 0.2720 - val_accuracy: 0.6899 - val_loss: 0.8509
Epoch 5/10
619/619 ━━━━━━━━━━━━━━━━━━━━ 43s 69ms/step - accuracy: 0.9284 - loss: 0.2014 - val_accuracy: 0.6926 - val_loss: 1.0663


In [36]:
from sklearn.metrics import classification_report, f1_score, accuracy_score

lstm_preds_prob = lstm_model.predict(X_test_pad)
lstm_preds = np.argmax(lstm_preds_prob, axis=1)

print("=== Bidirectional LSTM ===")
print(classification_report(y_test, lstm_preds, target_names=le.classes_))
print(f"Macro F1:    {f1_score(y_test, lstm_preds, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(y_test, lstm_preds, average='weighted'):.4f}")
print(f"Accuracy:    {accuracy_score(y_test, lstm_preds):.4f}")


172/172 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step
=== Bidirectional LSTM ===
              precision    recall  f1-score   support

    negative       0.75      0.66      0.70      1556
     neutral       0.68      0.65      0.67      2223
    positive       0.70      0.82      0.76      1717

    accuracy                           0.71      5496
   macro avg       0.71      0.71      0.71      5496
weighted avg       0.71      0.71      0.70      5496

Macro F1:    0.7081
Weighted F1: 0.7043
Accuracy:    0.7060


### Vanilla RNN — baseline comparison

In [37]:
from tensorflow.keras.layers import SimpleRNN, GRU, Bidirectional
from tensorflow.keras.optimizers import Adam

def build_rnn(units_1=128, units_2=64, dropout=0.3,
              bidirectional=True, rnn_type='GRU', lr=5e-4):
    """
    Fixed RNN. Key changes vs original:
    - Bidirectional: reads sequence in both directions (fixes collapsed predictions)
    - GRU by default: has gating like LSTM so it handles class imbalance better
      than SimpleRNN; use rnn_type='SimpleRNN' to compare
    - Lower lr (5e-4 vs 1e-3 Adam default): SimpleRNN/GRU train more stably
    """
    RNNLayer = GRU if rnn_type == 'GRU' else SimpleRNN

    layer_1 = RNNLayer(units_1, return_sequences=True)
    layer_2 = RNNLayer(units_2)

    if bidirectional:
        layer_1 = Bidirectional(layer_1)
        layer_2 = Bidirectional(layer_2)

    model = Sequential([
        Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),
        layer_1,
        Dropout(dropout),
        layer_2,
        Dropout(dropout),
        Dense(64, activation='relu'),
        Dense(3, activation='softmax')
    ])
    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=Adam(learning_rate=lr),
        metrics=['accuracy']
    )
    return model

# Train GRU (recommended fix)
print("=== Bidirectional GRU ===")
gru_model = build_rnn(rnn_type='GRU', bidirectional=True, lr=5e-4)
gru_model.summary()

history_gru = gru_model.fit(
    X_train_pad, y_train,
    epochs=10, batch_size=32,
    validation_split=0.1,
    class_weight=class_weight_dict,
    callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
    verbose=1
)

gru_preds = np.argmax(gru_model.predict(X_test_pad), axis=1)
print(classification_report(y_test, gru_preds, target_names=le.classes_))
print(f"Macro F1:    {f1_score(y_test, gru_preds, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(y_test, gru_preds, average='weighted'):.4f}")
print(f"Accuracy:    {accuracy_score(y_test, gru_preds):.4f}")

# Also try Bidirectional SimpleRNN for comparison
print("\n=== Bidirectional SimpleRNN (fixed) ===")
rnn_model = build_rnn(rnn_type='SimpleRNN', bidirectional=True, lr=5e-4)

history_rnn = rnn_model.fit(
    X_train_pad, y_train,
    epochs=10, batch_size=32,
    validation_split=0.1,
    class_weight=class_weight_dict,
    callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
    verbose=1
)

rnn_preds = np.argmax(rnn_model.predict(X_test_pad), axis=1)
print(classification_report(y_test, rnn_preds, target_names=le.classes_))
print(f"Macro F1:    {f1_score(y_test, rnn_preds, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(y_test, rnn_preds, average='weighted'):.4f}")
print(f"Accuracy:    {accuracy_score(y_test, rnn_preds):.4f}")


=== Bidirectional GRU ===


/opt/miniconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_18"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_18 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_22                │ ?                      │   0 (unbuilt) │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_36 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_23                │ ?                      │   0 (unbuilt) │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_37 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_36 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
619/619 ━━━━━━━━━━━━━━━━━━━━ 44s 65ms/step - accuracy: 0.5717 - loss: 0.8572 - val_accuracy: 0.6776 - val_loss: 0.7158
Epoch 2/10
619/619 ━━━━━━━━━━━━━━━━━━━━ 43s 69ms/step - accuracy: 0.7643 - loss: 0.5623 - val_accuracy: 0.7040 - val_loss: 0.7205
Epoch 3/10
619/619 ━━━━━━━━━━━━━━━━━━━━ 43s 70ms/step - accuracy: 0.8424 - loss: 0.3979 - val_accuracy: 0.6953 - val_loss: 0.7640
Epoch 4/10
619/619 ━━━━━━━━━━━━━━━━━━━━ 43s 69ms/step - accuracy: 0.8883 - loss: 0.2949 - val_accuracy: 0.6794 - val_loss: 0.9063
172/172 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step
              precision    recall  f1-score   support

    negative       0.67      0.67      0.67      1556
     neutral       0.66      0.62      0.64      2223
    positive       0.71      0.78      0.74      1717

    accuracy                           0.68      5496
   macro avg       0.68      0.69      0.68      5496
weighted avg       0.68      0.68      0.68      5496

Macro F1:    0.6829
Weighted F1: 0.6791
Accuracy:    0.68

/opt/miniconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


619/619 ━━━━━━━━━━━━━━━━━━━━ 20s 28ms/step - accuracy: 0.4355 - loss: 1.0221 - val_accuracy: 0.6057 - val_loss: 0.8487
Epoch 2/10
619/619 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.6876 - loss: 0.7062 - val_accuracy: 0.6603 - val_loss: 0.7789
Epoch 3/10
619/619 ━━━━━━━━━━━━━━━━━━━━ 19s 31ms/step - accuracy: 0.8244 - loss: 0.4458 - val_accuracy: 0.6553 - val_loss: 0.9078
Epoch 4/10
619/619 ━━━━━━━━━━━━━━━━━━━━ 18s 30ms/step - accuracy: 0.9194 - loss: 0.2249 - val_accuracy: 0.6453 - val_loss: 1.0951
Epoch 5/10
619/619 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.9533 - loss: 0.1323 - val_accuracy: 0.6194 - val_loss: 1.3456
172/172 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
              precision    recall  f1-score   support

    negative       0.68      0.62      0.65      1556
     neutral       0.61      0.74      0.67      2223
    positive       0.80      0.64      0.71      1717

    accuracy                           0.67      5496
   macro avg       0.70      0.67      0.68     

In [38]:
# GRU v2 — fix overfitting with higher dropout + larger batch
gru_model_v2 = build_rnn(
    units_1=64,
    units_2=32,
    dropout=0.5,
    rnn_type='GRU',
    bidirectional=True,
    lr=5e-4
)

history_gru_v2 = gru_model_v2.fit(
    X_train_pad, y_train,
    epochs=10, batch_size=64,
    validation_split=0.1,
    class_weight=class_weight_dict,
    callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
    verbose=1
)

gru_v2_preds = np.argmax(gru_model_v2.predict(X_test_pad), axis=1)
print(classification_report(y_test, gru_v2_preds, target_names=le.classes_))
print(f"Macro F1:    {f1_score(y_test, gru_v2_preds, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(y_test, gru_v2_preds, average='weighted'):.4f}")
print(f"Accuracy:    {accuracy_score(y_test, gru_v2_preds):.4f}")

Epoch 1/10


/opt/miniconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


310/310 ━━━━━━━━━━━━━━━━━━━━ 19s 49ms/step - accuracy: 0.5185 - loss: 0.9343 - val_accuracy: 0.6389 - val_loss: 0.7867
Epoch 2/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 17s 54ms/step - accuracy: 0.7361 - loss: 0.6290 - val_accuracy: 0.6598 - val_loss: 0.7863
Epoch 3/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 17s 54ms/step - accuracy: 0.8254 - loss: 0.4575 - val_accuracy: 0.6639 - val_loss: 0.7924
Epoch 4/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 17s 54ms/step - accuracy: 0.8763 - loss: 0.3376 - val_accuracy: 0.6617 - val_loss: 0.9292
Epoch 5/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 17s 55ms/step - accuracy: 0.9072 - loss: 0.2671 - val_accuracy: 0.6498 - val_loss: 1.0968
172/172 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step
              precision    recall  f1-score   support

    negative       0.74      0.60      0.66      1556
     neutral       0.65      0.56      0.60      2223
    positive       0.62      0.85      0.71      1717

    accuracy                           0.66      5496
   macro avg       0.67      0.67      0.66     

### Hyperparam Sweep — LSTM & RNN (Macro F1)

In [39]:
sweep_results = []

configs = [
    # (model_type, units_1, units_2, dropout)
    ("LSTM",  64,  32,  0.2),
    ("LSTM",  64,  32,  0.4),
    ("LSTM",  128, 64,  0.2),
    ("LSTM",  128, 64,  0.4),   # baseline
    ("LSTM",  256, 128, 0.2),
    ("LSTM",  256, 128, 0.4),
    ("RNN",   64,  32,  0.2),
    ("RNN",   64,  32,  0.4),
    ("RNN",   128, 64,  0.2),
    ("RNN",   128, 64,  0.4),   # baseline
    ("RNN",   256, 128, 0.2),
    ("RNN",   256, 128, 0.4),
]

for model_type, u1, u2, dr in configs:
    run_name = f"{model_type}_u{u1}-{u2}_dr{dr}"
    print(f"Running: {run_name}")

    m = build_lstm(u1, u2, dr, bidirectional=True) if model_type == "LSTM" else build_rnn(u1, u2, dr)

    m.fit(
        X_train_pad, y_train,
        epochs=10, batch_size=32, validation_split=0.1,
        class_weight=class_weight_dict,
        callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
        verbose=0
    )

    preds = np.argmax(m.predict(X_test_pad, verbose=0), axis=1)

    sweep_results.append({
        "Model":       run_name,
        "Type":        model_type,
        "Units":       f"{u1}-{u2}",
        "Dropout":     dr,
        "Macro F1":    round(f1_score(y_test, preds, average='macro'),    4),
        "Weighted F1": round(f1_score(y_test, preds, average='weighted'), 4),
        "Accuracy":    round(accuracy_score(y_test, preds), 4),
    })
    print(f"  -> Macro F1: {sweep_results[-1]['Macro F1']} | Acc: {sweep_results[-1]['Accuracy']}")

sweep_df = pd.DataFrame(sweep_results).sort_values("Macro F1", ascending=False)
print("\n=== HYPERPARAM SWEEP (sorted by Macro F1) ===")
print(sweep_df.to_string(index=False))


Running: LSTM_u64-32_dr0.2


/opt/miniconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


  -> Macro F1: 0.7132 | Acc: 0.7096
Running: LSTM_u64-32_dr0.4


/opt/miniconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


  -> Macro F1: 0.7051 | Acc: 0.7009
Running: LSTM_u128-64_dr0.2


...

  -> Macro F1: 0.6969 | Acc: 0.6936
Running: RNN_u256-128_dr0.2


/opt/miniconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


  -> Macro F1: 0.7102 | Acc: 0.7061
Running: RNN_u256-128_dr0.4


/opt/miniconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


  -> Macro F1: 0.6988 | Acc: 0.6945

=== HYPERPARAM SWEEP (sorted by Macro F1) ===
              Model Type   Units  Dropout  Macro F1  Weighted F1  Accuracy
  LSTM_u64-32_dr0.2 LSTM   64-32      0.2    0.7132       0.7100    0.7096
LSTM_u256-128_dr0.2 LSTM 256-128      0.2    0.7105       0.7080    0.7078
 RNN_u256-128_dr0.2  RNN 256-128      0.2    0.7102       0.7065    0.7061
 LSTM_u128-64_dr0.4 LSTM  128-64      0.4    0.7083       0.7035    0.7034
 LSTM_u128-64_dr0.2 LSTM  128-64      0.2    0.7079       0.7048    0.7043
  LSTM_u64-32_dr0.4 LSTM   64-32      0.4    0.7051       0.7005    0.7009
   RNN_u64-32_dr0.2  RNN   64-32      0.2    0.6989       0.6935    0.6963
 RNN_u256-128_dr0.4  RNN 256-128      0.4    0.6988       0.6923    0.6945
  RNN_u128-64_dr0.4  RNN  128-64      0.4    0.6969       0.6932    0.6936
  RNN_u128-64_dr0.2  RNN  128-64      0.2    0.6946       0.6898    0.6912
LSTM_u256-128_dr0.4 LSTM 256-128      0.4    0.6773       0.6720    0.6769
   RNN_u64-32_dr0

In [40]:
!pip install transformers datasets torch

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, get_scheduler
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score, accuracy_score

In [41]:
bert_tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(texts, labels, max_len=64):
    encodings = bert_tokenizer(
        list(texts),
        truncation=True,
        padding=True,
        max_length=max_len,
        return_tensors='pt'
    )
    return encodings, torch.tensor(list(labels), dtype=torch.long)

print("Tokenizing...")
train_encodings, train_labels_tensor = tokenize(X_train, y_train)
test_encodings,  test_labels_tensor  = tokenize(X_test,  y_test)
print("Done!")

Tokenizing...
Done!


In [42]:
class TweetDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = TweetDataset(train_encodings, train_labels_tensor)
test_dataset  = TweetDataset(test_encodings,  test_labels_tensor)

print(f"Train size: {len(train_dataset)}")
print(f"Test size:  {len(test_dataset)}")

Train size: 21984
Test size:  5496


In [43]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def build_distilbert(unfreeze_layers=0):
    """
    Load DistilBERT and selectively unfreeze transformer layers.
    DistilBERT has 6 transformer layers total.
    unfreeze_layers=0 → only classification head trains (frozen transformer)
    unfreeze_layers=6 → all layers fine-tune (full model)
    """
    model = DistilBertForSequenceClassification.from_pretrained(
        'distilbert-base-uncased',
        num_labels=3
    )

    # Step 1: freeze everything
    for param in model.distilbert.parameters():
        param.requires_grad = False

    # Step 2: unfreeze the last n transformer layers
    if unfreeze_layers > 0:
        for layer in model.distilbert.transformer.layer[-unfreeze_layers:]:
            for param in layer.parameters():
                param.requires_grad = True

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"u{unfreeze_layers} — Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

    return model.to(device)

Using device: cpu


In [44]:
def train_model(model, train_dataset, num_epochs=3, batch_size=32):
    """Fine-tune a model and return it."""
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5)
    total_steps = num_epochs * len(train_loader)
    scheduler = get_scheduler("linear", optimizer=optimizer,
                               num_warmup_steps=0, num_training_steps=total_steps)

    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        correct = 0
        total   = 0

        for batch in train_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss    = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

        print(f"  Epoch {epoch+1}/{num_epochs} — Loss: {total_loss/len(train_loader):.4f} — Acc: {correct/total:.4f}")

    return model

In [45]:
def evaluate_model(model, test_dataset, batch_size=32):
    """Run inference and return predictions."""
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.array(all_preds), np.array(all_labels)

In [46]:
experiment_results = {}

for n_layers in [0, 2, 4, 6]:
    run_name = f"distilbert_u{n_layers}"
    print(f"\n{'='*50}\nRunning: {run_name}\n{'='*50}")

    model = build_distilbert(unfreeze_layers=n_layers)
    model = train_model(model, train_dataset, num_epochs=3, batch_size=32)
    preds, labels = evaluate_model(model, test_dataset)

    macro_f1    = f1_score(labels, preds, average='macro')
    weighted_f1 = f1_score(labels, preds, average='weighted')
    acc         = accuracy_score(labels, preds)

    experiment_results[run_name] = {
        "Macro F1":    round(macro_f1, 4),
        "Weighted F1": round(weighted_f1, 4),
        "Accuracy":    round(acc, 4)
    }

    print(f"\n{run_name} -> Macro F1: {macro_f1:.4f} | Acc: {acc:.4f}")
    print(classification_report(labels, preds, target_names=le.classes_))



Running: distilbert_u0


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6521.50it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


u0 — Trainable params: 592,899 / 66,955,779 (0.9%)
  Epoch 1/3 — Loss: 1.0252 — Acc: 0.4814
  Epoch 2/3 — Loss: 0.9356 — Acc: 0.5668
  Epoch 3/3 — Loss: 0.9045 — Acc: 0.5844

distilbert_u0 -> Macro F1: 0.5881 | Acc: 0.5928
              precision    recall  f1-score   support

    negative       0.66      0.46      0.54      1556
     neutral       0.53      0.69      0.60      2223
    positive       0.66      0.59      0.62      1717

    accuracy                           0.59      5496
   macro avg       0.62      0.58      0.59      5496
weighted avg       0.61      0.59      0.59      5496


Running: distilbert_u2


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 13948.93it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


u2 — Trainable params: 14,768,643 / 66,955,779 (22.1%)
  Epoch 1/3 — Loss: 0.6885 — Acc: 0.6974
  Epoch 2/3 — Loss: 0.5651 — Acc: 0.7654
  Epoch 3/3 — Loss: 0.5349 — Acc: 0.7796

distilbert_u2 -> Macro F1: 0.7702 | Acc: 0.7667
              precision    recall  f1-score   support

    negative       0.76      0.78      0.77      1556
     neutral       0.73      0.74      0.73      2223
    positive       0.83      0.80      0.81      1717

    accuracy                           0.77      5496
   macro avg       0.77      0.77      0.77      5496
weighted avg       0.77      0.77      0.77      5496


Running: distilbert_u4


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 10316.82it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


u4 — Trainable params: 28,944,387 / 66,955,779 (43.2%)
  Epoch 1/3 — Loss: 0.6274 — Acc: 0.7340
  Epoch 2/3 — Loss: 0.4920 — Acc: 0.8030
  Epoch 3/3 — Loss: 0.4407 — Acc: 0.8238

distilbert_u4 -> Macro F1: 0.7884 | Acc: 0.7851
              precision    recall  f1-score   support

    negative       0.78      0.80      0.79      1556
     neutral       0.76      0.75      0.75      2223
    positive       0.82      0.82      0.82      1717

    accuracy                           0.79      5496
   macro avg       0.79      0.79      0.79      5496
weighted avg       0.79      0.79      0.79      5496


Running: distilbert_u6


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 13290.36it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


u6 — Trainable params: 43,120,131 / 66,955,779 (64.4%)
  Epoch 1/3 — Loss: 0.5999 — Acc: 0.7528
  Epoch 2/3 — Loss: 0.4555 — Acc: 0.8199
  Epoch 3/3 — Loss: 0.3822 — Acc: 0.8539

distilbert_u6 -> Macro F1: 0.7942 | Acc: 0.7906
              precision    recall  f1-score   support

    negative       0.79      0.80      0.80      1556
     neutral       0.76      0.76      0.76      2223
    positive       0.83      0.83      0.83      1717

    accuracy                           0.79      5496
   macro avg       0.79      0.79      0.79      5496
weighted avg       0.79      0.79      0.79      5496



In [47]:
best_lstm = sweep_df[sweep_df['Type'] == 'LSTM'].iloc[0]
best_rnn  = sweep_df[sweep_df['Type'] == 'RNN'].iloc[0]

all_results = {
    "Random Forest": {
        "Macro F1":    round(f1_score(y_test, rf_preds,   average='macro'),    4),
        "Weighted F1": round(f1_score(y_test, rf_preds,   average='weighted'), 4),
        "Accuracy":    round(accuracy_score(y_test, rf_preds), 4)
    },
    "XGBoost": {
        "Macro F1":    round(f1_score(y_test, xgb_preds,  average='macro'),    4),
        "Weighted F1": round(f1_score(y_test, xgb_preds,  average='weighted'), 4),
        "Accuracy":    round(accuracy_score(y_test, xgb_preds), 4)
    },
    f"LSTM best ({best_lstm['Model']})": {
        "Macro F1":    best_lstm["Macro F1"],
        "Weighted F1": best_lstm["Weighted F1"],
        "Accuracy":    best_lstm["Accuracy"],
    },
    f"RNN best ({best_rnn['Model']})": {
        "Macro F1":    best_rnn["Macro F1"],
        "Weighted F1": best_rnn["Weighted F1"],
        "Accuracy":    best_rnn["Accuracy"],
    },
}
all_results.update(experiment_results)

summary = pd.DataFrame(all_results).T.sort_values("Macro F1", ascending=False)
print("=== FINAL MODEL COMPARISON (sorted by Macro F1) ===")
print(summary.to_string())


=== FINAL MODEL COMPARISON (sorted by Macro F1) ===
                               Macro F1  Weighted F1  Accuracy
distilbert_u6                    0.7942       0.7906    0.7906
distilbert_u4                    0.7884       0.7851    0.7851
distilbert_u2                    0.7702       0.7670    0.7667
LSTM best (LSTM_u64-32_dr0.2)    0.7132       0.7100    0.7096
RNN best (RNN_u256-128_dr0.2)    0.7102       0.7065    0.7061
XGBoost                          0.6962       0.6972    0.6976
Random Forest                    0.6762       0.6784    0.6801
distilbert_u0                    0.5881       0.5906    0.5928
